# GlycoQueue: Capacity-Aware Prioritisation for HbA1c Screening

Hello, and thank you for reading this!

**Track** AI + Health: (custom, not built from suggested projects) 
**Team:** Solo

**One-sentence pitch:**  GlycoQueue helps primary care coordinators decide who should be recalled for HbA1c tests when capacity is limited, while maintaining an age-diverse reach, to maximise the positive cases of elevated HbA1c (a sign of diabetes) caught in screening

**How to run:** simply download the [GlycoQuece.ipynb](GlycoQuece.ipynb) notebook, open it in Colab, and run by Runtime > Run All
The notebook contains explanations and a concise clear progression within it, but to see a more detailed and verbose walkthrough and my thought process/reasoning on why or how I did some things, feel free to read through the  [GlycoQuece_walkthrough_verbose.ipynb](GlycoQuece_walkthrough_verbose.ipynb), but the [GlycoQuece.ipynb](GlycoQuece.ipynb) notebook is sufficient and contains all necessary information and context

**DISCLAIMER: this project is a PROTOTYPE and is NOT medical advice or clinical diagnosis**

**We will investigate**: What if we have a hypothetical scenario: 

We are the coordinators responsible for screening this cohort of patients for a blood test, specifically testing for the HbA1c test, however, we only have enough supply for 10% of people, or 15%, or n% where it is a limited supply for a larger demand. How can we make a model that can capture as many positive tests as possible?

Some further context about the HbA1c test, which I could not have said better myself:

"
The hemoglobin A1c test — also known as glycated hemoglobin, glycosylated hemoglobin, HbA1c, or simply A1c—is used to measure an individual's glucose control levels. The test shows average blood sugar levels over the past 90 days, expressed as a percentage. In addition, it can be used to diagnose diabetes mellitus.[1] Hemoglobin is a protein found exclusively in red blood cells, giving blood its bright red color. The primary role of hemoglobin is to carry oxygen from the lungs to all the cells in the body. Hemoglobin becomes glycated or coated with glucose from the bloodstream. As blood glucose levels increase, more glucose attaches to the hemoglobin protein, resulting in a higher A1c value.[2] Since red blood cells have an average lifespan of about 3 months, the A1c test measures hemoglobin levels in the bloodstream over this period, making it a reliable indicator of blood sugar control.
"

(source: Basu S, et al. Hemoglobin A1C. StatPearls. 2025. Available from: https://www.ncbi.nlm.nih.gov/sites/books/NBK549816/)

In [ ]:
### This cell is about importing libraries

from io import BytesIO # we handle government data in .xpt - which is a legacy format which would otherwise require specific tools like SAS to read
# bytesio makes these downloaded data behave like a file object so panda can read this off the internet directly in memory, without downloading or saving to memory to treat like a file
from pathlib import Path # generally useful for file path manipulation and to create output folders easily avoiding crashes
import time # since we are downloading data off the internet, we need this for our backoff download func 
import requests # easily handle downloading our government data off the internet
import numpy as np # for numerical operations especially with pandas and arrays
import pandas as pd # for data manipulation and analysis
import matplotlib.pyplot as plt # to plot graphs to visualise the data our model sees

from sklearn.compose import ColumnTransformer # as shown in the code later, this is to manipulate columns at once
from sklearn.impute import SimpleImputer # some fields in the NHANES is missing (NaN) - we need to impute them
from sklearn.preprocessing import StandardScaler # this is the second part needed later in our colummn transformer
from sklearn.pipeline import Pipeline # to make pipelines, standard stuff - so that things happen neatly in order
from sklearn.linear_model import LogisticRegression # our selected model - this is all we need for a 2d input and a classifier output wtih a decision tree
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
# These three metrics will help us assess our models, allowing us to compare with a baseline (random allocation)
# This is further explained in the md cell below this

import joblib # to save our model to disk and load it back up again later, so we don't have to retrain it every time we want to use it

### Metrics explained

- ROC-AUC measures ranking ability, on how often approximately the model can distinguish and rank a positive result above a negative
- Avg Precision: Meausres reliability of positive predictions (Is the "high risk" indicators correct?)
- Brier score: this measures overconfidence - called "calibration" ("when the model is confident, is it right? this punished overconfidence and exposes lack of learning and generalising)

Critically, **accuracy alone is NOT sufficient** - as a model simply guessing negative will be right most of the time, because most people dont have diabetes. However, this model is completely *useless* and learns nothing, so this disproves that accuracy is sufficient for model grading

### Evaluation reasoning:

*Why don't we aim for 100%?* Diabetes is a complex condition that is caused by a variety including genetics, diet, and environment, beyond simply age and BMI, so these factors alone cannot accurately diagnose diabetes. Therefore, a model only using age and BMI CANNOT and SHOULD NOT be expected to achieve 100% classification.

My real target is **"Lift"** over randomness. Instead of chasing perfect classification we measure **"Lift"** which is how much better our model and policy performs compared to the minimum baseline (which would be random selection)

Even if random screening finds 5 case in 10 tests, and our model manages to catch 8 positive cases out of its predicted 10, then that would be **x1.6 Lift**, 1.6 times the clinical efficiency! In resource constrained healthcare, such efficiency gains make a big difference in impact of positive cases caught in screening using the **same** budget. 

Success is about identifying high-risk individuals significantly better than random guessing/chance, *not* about predicting every case perfectly with limited factors like age and BMI.



In [ ]:
### This cell is about defining constants and settings for our model

CYCLES = {
    "2015-2016": {
        "folder": "2015", "suffix": "I", "weight": "WTMEC2YR",
        "components": ["DEMO", "BMX", "DIQ", "GHB"]
    },
    "2017-2018": {
        "folder": "2017", "suffix": "J", "weight": "WTMEC2YR",
        "components": ["DEMO", "BMX", "DIQ", "GHB"]
    },
    "2021-2023": {
        "folder": "2021", "suffix": "L", "weight": "WTPH2YR",
        "components": ["DEMO", "BMX", "DIQ", "GHB"]
    },
}

# These are NHANES labels in their database because their headers were restricted to 8 chars by .xpt

# DEMO = Demographics like age (RIDAGEYR), sex (RIAGENDR), race (RIDRETH1), and pregnancy status (RIDEXPRG) 
# to make the sample population representative, it is stratified, and the weights are provided in columns (WTMEC2YR, WTPH2YR)

# BMX = Body Measures like height (BMXHT), weight (BMXWT), and BMI (BMXBMI)

# DIQ = Diabetes Questionnaire like diabetes diagnosis (DIQ010 - ever told you had diabetes?), (DIQ160 - same but prediabetes?), (DIQ180) - recent blood test
# This is needed for eligibility filters later:
# we want our model to estimate who had diabetes from their blood test, but DIDNT know it beforehand
# theres no point making a model telling diabetic people they have diabetes 

# GHB = Glycohemoglobin (HbA1c) - this is the blood test result that tells us if someone has elevated HbA1c levels, which is a sign of diabetes, NOT a diagnosis

# here we simply define some top level constants for our model
TRAIN_CYCLE = "2015-2016"
VALID_CYCLE = "2017-2018"
TEST_CYCLE = "2021-2023"
TARGET_THRESHOLD = 5.7 # for elevated HbA1c
PRIMARY_CAPACITY = 0.10 # percentage of the population that can be screened
FEATURES = ["AGE", "BMI"] # the columns we will feed the model as input
PRIMARY_RESERVE = 0.30 # percentage of the testing capacity reserved for younger adults aged<50 
SECONDARY_RESERVE = 0.45 # same but 45%
OUTPUT_DIR = Path("glycoqueue_final_test") # to store results for reproducibility
OUTPUT_DIR.mkdir(exist_ok=True) # nice because it avoids crashing if it exists already, and creates it if it doesn't exist

BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{folder}/DataFiles/{component}_{suffix}.xpt"

## settings have been chosen, and 2021-2023 is unlocked in this final version
## it will be used for testing and evaluation, not tuning or training

In [ ]:
### This cell is about downloading the dataset and merging it so its a usable dataframe for our model.
### The NHANES stores different data like age, BMI in their own massive files, and we merge them together to get unified records per participant with their data

session = requests.Session() # reusable connection
session.headers.update({"User-Agent": "GlycoQueue-Research/1.0 (public NHANES analysis)"})

def read_xpt(cycle_name, component, max_retries=3):
    meta = CYCLES[cycle_name]
    url = BASE_URL.format(folder=meta["folder"], component=component, suffix=meta["suffix"])
    # above here, the cycles are built from the cycles dictionary, and the url is built from the base url and the cycle metadata

    last_error = None # robustness
    for attempt in range(1, max_retries + 1): # try up to three times
        try:
            response = session.get(url, timeout=90)
            response.raise_for_status() # throw an exception if the download failed
            frame = pd.read_sas(BytesIO(response.content), format="xport", encoding="utf-8")
            frame.columns = [str(c).upper() for c in frame.columns] # covert it to a dataframe

            # must ensure data is not corrupt, and that the SEQN - which is the unique patient identifier - is present and valid, otherwise the merge will fail LOUDLY - shouldnt happen tho, i tried it
            if "SEQN" not in frame.columns:
                raise ValueError(f"{component} in {cycle_name} has no SEQN column")
            if frame["SEQN"].isna().any() or frame["SEQN"].duplicated().any():
                raise ValueError(f"{component} in {cycle_name} has missing or duplicated SEQN values")
            return frame

        except Exception as exc: #retry if failed, once again, shouldnt happen but just in case - its confirmed to work by me already
            last_error = exc
            print(f"{cycle_name} {component}: attempt {attempt}/{max_retries} failed: {exc}")
            if attempt < max_retries:
                time.sleep(2 * attempt)
    raise RuntimeError(f"Could not download {cycle_name} {component}") from last_error


# step 2: merging - the NHANES stores different data like age, BMI in their own massive files, and we need to merge them together to get records per participant with their BMI, age, etc unified 
merged_cycles = {}
merge_rows = []
for cycle_name, meta in CYCLES.items():
    parts = {component: read_xpt(cycle_name, component) for component in meta["components"]}

    # start from demographics because all the participants are in it
    merged = parts["DEMO"].copy()
    for component in ["BMX", "DIQ", "GHB"]:
        merged = merged.merge(
            parts[component], on="SEQN", how="left", validate="one_to_one"
        )
        # merge bmx, diq, and ghb to demo on SEQN, which is the unique participant identifier, and validate that the merge is one-to-one before merging
        # how left means that if a participant is missing data in bmx, diq, or ghb, they will still be included in the merged dataframe, but with NaN values for the missing data
        # that is important to keep the sample size large and to count missing data later

        # for auditing, record which cycle, which component, rows and unique participants, so there is proof that no proof silently duplicated or dropped anyone. 
        # check: if (rows_after_merge != unique_participant_ids) for any row, then something went wrong, error
        merge_rows.append({
            "cycle": cycle_name,
            "component_added": component,
            "rows_after_merge": len(merged),
            "unique_participant_ids": merged["SEQN"].nunique(),
        })
    merged["CYCLE"] = cycle_name # make sure data is labelled by the cycle it came from, we want the seperate batches of cohorts to be separable
    merged["HBA1C_WEIGHT"] = pd.to_numeric(merged[meta["weight"]], errors="coerce") # find and replace all weights with their correct weight depending on the cycle
    # what is the weight? btw, the weight is so the sample that the NHANES collects is representative of the US population, 
    # because they oversample certain groups like minorities and the elderly, and undersample others like young healthy adults. 
    # The weight is used to correct for this sampling bias when analyzing the data, and its for who was selected for the blood tests
    merged_cycles[cycle_name] = merged
    print(f"{cycle_name}: {len(merged):,} rows; {merged['SEQN'].nunique():,} unique participants") # output so the user can see the results of the merge 

merge_audit = pd.DataFrame(merge_rows)
display(merge_audit)
merge_audit.to_csv(OUTPUT_DIR / "merge_audit.csv", index=False) # output and save to a file

In [ ]:
## THis cell is about filtering our dataset to only extract valid ADULTS who are NOT pregnant, and Have NOT reported diabetes or prediabetes, and have a valid HbA1c blood test result

cohorts = {} # this will hold the final cohorts for each cycle, after filtering for eligibility and adding the target variable
flow_rows = [] # holding audit tables for how many survived each filter
for cycle_name, raw in merged_cycles.items():
    df = raw.copy() # opying is always beffer than mutation - rust lessons learned

    # rename the columns to more human readable names, and convert them to numeric where appropriate, and create age groups for analysis
    # fun fact: the NHANES data is stored in a legacy format called .xpt, which was originally designed for SAS, and has a limit of 8 characters for column names.
    # thats why they are so unreadable 
    df["AGE"] = pd.to_numeric(df["RIDAGEYR"], errors="coerce")
    df["BMI"] = pd.to_numeric(df["BMXBMI"], errors="coerce")
    df["SEX"] = df["RIAGENDR"].map({1: "Male", 2: "Female"}).fillna("Unknown / missing")
    df["ETHNICITY"] = df["RIDRETH3"].map({
        1: "Mexican American",
        2: "Other Hispanic",
        3: "Non-Hispanic White",
        4: "Non-Hispanic Black",
        6: "Non-Hispanic Asian", # yes 5 is intentionally skipped for some reason, probably a legacy issue
        7: "Other race / multiracial",
    }).fillna("Unknown / missing")
    df["AGE_GROUP"] = pd.cut(
        df["AGE"], bins=[17, 34, 49, 64, 150],
        labels=["18–34", "35–49", "50–64", "65+"], include_lowest=True
    )

    # define eligibility criteria for the cohort: adults (18+), no reported diabetes, no reported prediabetes, has valid HbA1c result, and not known to be pregnant
    adult = df["AGE"].ge(18)
    no_diabetes = df["DIQ010"].eq(2)
    no_prediabetes = df["DIQ160"].eq(2)
    has_hba1c = pd.to_numeric(df["LBXGH"], errors="coerce").notna()
    known_pregnant = df["RIAGENDR"].eq(2) & df["RIDEXPRG"].eq(1)
    eligible = adult & no_diabetes & no_prediabetes & has_hba1c & ~known_pregnant

    # record the flow of participants through the eligibility criteria for auditing and transparency
    flow_rows.append({
        "cycle": cycle_name,
        "all_demographic_records": len(df),
        "adults_18_plus": int(adult.sum()),
        "no_reported_diabetes": int((adult & no_diabetes).sum()),
        "no_reported_diabetes_or_prediabetes": int((adult & no_diabetes & no_prediabetes).sum()),
        "with_hba1c": int((adult & no_diabetes & no_prediabetes & has_hba1c).sum()),
        "eligible_after_known_pregnancy_exclusion": int(eligible.sum()), # keep in mind only an age range of females were asked on pregnancy, and this is REPORTED, not KNOWN and TESTED, hence the cautious naming
        "elevated_hba1c_ge_5_7": int((eligible & (pd.to_numeric(df["LBXGH"], errors="coerce") >= TARGET_THRESHOLD)).sum()), # these are more of quality checking so we see what we work with
        "diabetes_range_ge_6_5_secondary": int((eligible & (pd.to_numeric(df["LBXGH"], errors="coerce") >= 6.5)).sum()),
        "missing_or_nonpositive_weight_in_eligible": int((eligible & (df["HBA1C_WEIGHT"].isna() | df["HBA1C_WEIGHT"].le(0))).sum()),
    })

    # build cohort dataframe for the eligible participants, and add the target variable (elevated HbA1c >= 5.7)
    cohort = df.loc[eligible].copy().reset_index(drop=True)
    if cohort.empty: # fail loudly
        raise ValueError(f"No eligible participants in {cycle_name}")
    if cohort["HBA1C_WEIGHT"].isna().any() or (cohort["HBA1C_WEIGHT"] <= 0).any():
        raise ValueError(f"Missing/non-positive analysis weights in eligible {cycle_name} cohort")
    cohort["TARGET"] = (pd.to_numeric(cohort["LBXGH"], errors="coerce") >= TARGET_THRESHOLD).astype(int)
    cohorts[cycle_name] = cohort

cohort_flow = pd.DataFrame(flow_rows)
display(cohort_flow)
cohort_flow.to_csv(OUTPUT_DIR / "cohort_flow.csv", index=False) # for auditing and reproducibility

for cycle_name, cohort in cohorts.items():
    print(
        f"{cycle_name}: n={len(cohort):,}; positives={int(cohort['TARGET'].sum()):,}; "
        f"unweighted prevalence={cohort['TARGET'].mean():.1%}; "
        f"weighted prevalence={(np.average(cohort['TARGET'], weights=cohort['HBA1C_WEIGHT'])):.1%}"
    )

In [ ]:
## This cell is about training the model on the 2015-2016 cohort, and scoring the validation and test cohorts with it.
# assign the data frames now
train_df = cohorts[TRAIN_CYCLE] 
valid_df = cohorts[VALID_CYCLE]
test_df = cohorts[TEST_CYCLE]

# x = input, y = outcome to be trained for, in this case, the target variable is elevated HbA1c >= 5.7, which is a binary classification problem
y_train = train_df["TARGET"].astype(int).to_numpy()
# "TARGET" is the column we created earlier in the cohort building step, which is 1 if the participant has elevated HbA1c >= 5.7, and 0 otherwise
# to_numpy means we convert the pandas series to a numpy array, which is what sklearn expects for training

# cleaning the data and scaling it before training the model
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([ 
        ("imputer", SimpleImputer(strategy="median")), # we calculate the median and replace all missing values with it
        ("scaler", StandardScaler()), # scale to a consistent range about a mean of 0 and standard deviation of 1, important maths stuff - so elegantly simple and interestingly critical and helps basically allow the whole thing to work
    ]), FEATURES), # the features we ended up using was Age and BMI - the rest provided minimal predictive value and were dropped to keep the model interpretable and simple
], remainder="drop") # after experimenting using a variety mix and match of features, i found that age and bmi were the only ones worth keeping

model = Pipeline([
    ("preprocess", preprocessor), # prepare data
    ("model", LogisticRegression(max_iter=3000, solver="lbfgs")), # use a very smart algorithm - the standard one to train the logistic regression model
])
model.fit(train_df[FEATURES], y_train) # train the model ONLY on the training cohort, mapping age and BMI to the target variable of elevated HbA1c >= 5.7 (binary classification)

# Score all eligible participants. HbA1c labels are not model inputs.
score_frames = {}
for cycle_name in [VALID_CYCLE, TEST_CYCLE]:
    frame = cohorts[cycle_name]
    score_frames[cycle_name] = model.predict_proba(frame[FEATURES])[:, 1]

joblib.dump(model, OUTPUT_DIR / "locked_age_bmi_logistic.joblib") # save the trained model for many useful reasons
print("Model fitted on 2015–2016. Validation and final-test cohorts have been scored.")
print("Model features:", FEATURES)

In [ ]:
### This cell is about defining the allocation policies and metrics for evaluation

# boolean mask of the top k scores from a list of scores, given a list of indices to consider, and the total number of scores
# this is our general function/tool to just get the top k in any little list we pass to it
def top_k_mask(scores, indices, k, n_total):
    selected = np.zeros(n_total, dtype=bool)
    indices = np.asarray(indices, dtype=int)
    k = int(max(0, min(k, len(indices)))) # clamper to ensure k is valid and possible
    if k: # as long as k is not 0 (invalidated by clamp) # sort, highest first, and select the top k indices
        order = np.argsort(-np.asarray(scores)[indices], kind="mergesort")
        selected[indices[order[:k]]] = True
    return selected

def global_policy(scores, k):
    return top_k_mask(scores, np.arange(len(scores)), k, len(scores)) # simple, select top k at risk without stratification - this leads to age bias towards older patients

# this splits k slots into proportional groups, then floors then gives leftovers to biggest fractions so totals add up to k, rounding can lose information and not work
# this is basically a helper function to split apart a population sized k into age stratified quotes per range, its not a policy itself
def proportional_age_quotas(frame, k):
    groups = ["18–34", "35–49", "50–64", "65+"]
    counts = frame["AGE_GROUP"].value_counts(sort=False).reindex(groups, fill_value=0) # this will basically split the given array into a count for each age range
    raw = counts.to_numpy(dtype=float) * k / len(frame) # calculate ideal proportion this age group needs
    quotas = np.floor(raw).astype(int) # round down to the nearest integer, this will usually undershoot
    remainder = k - int(quotas.sum()) # find overshoot
    fractions = raw - quotas # which group lost the most to rounding

    # for index in the fractions array which is sorted on which groups lost the most, and their indexes returned
    # THEN take the first {remainder} positions, which are the group positions with the two largest fractions 
    # then give each of these groups one extra slot to allocate the leftovers to those who deserve it most (was robbed the most by roubding down)
    for idx in np.argsort(-fractions, kind="mergesort")[:remainder]: 
        quotas[idx] += 1 # this is where its adding one to the indexes that lost most and therefore deserve it most
    return dict(zip(groups, quotas.tolist())) # zip the group names with the final quotes per age range


# the first policy, giving each age group its quota of slots filling with the highest risk people within that group
def age_stratified_policy(frame, scores, k):
    selected = np.zeros(len(frame), dtype=bool) # start with an all false mask for the full cohort, we will build who gets selected
    quotas = proportional_age_quotas(frame, k) # get quotes from calling function above
    age_groups = frame["AGE_GROUP"].astype("object").to_numpy() # extract the age group labels into a numpy array
    for group, quota in quotas.items():
        idx = np.flatnonzero(age_groups == group) # get the indexes for the rows in the current group from the wider pool
        selected |= top_k_mask(scores, idx, quota, len(frame)) # select for quota amount for this group
    if int(selected.sum()) != int(k): # if the math went wrong an the k number of masks selected doesnt match the selected actual amount 
        raise AssertionError(f"Age-stratified selection count {selected.sum()} != {k}; quotas={quotas}")
    return selected # return the final mask


The idea: a pure global policy barely picks young people because they score lower (as age lower = lower risk generally), and a pure age-stratified policy picks too few high risk older people (because it forces proportional representation, even though diabetes is not evenly distributed). This hybrid policy tries to increase **age-diverse reach** without giving up overall detection too much:

In [ ]:

def hybrid_under50_policy(frame, scores, k, reserved_share): # reserved share = fraction of capacity for testing withheld for younger adults
    ages = frame["AGE"].to_numpy(dtype=float)
    under50_idx = np.flatnonzero(ages < 50) # row numbers of under 50s
    reserve_k = int(round(k * reserved_share)) # how many to reserve
    reserve_k = min(reserve_k, len(under50_idx), k) # clamp
    selected = top_k_mask(scores, under50_idx, reserve_k, len(frame)) # NOW pick highest scoring under 50s
    remaining_idx = np.flatnonzero(~selected) # fill the rest of the slots globally by risk prob
    fill_k = k - int(selected.sum()) # how many are still empty
    selected |= top_k_mask(scores, remaining_idx, fill_k, len(frame)) # "OR" fill the rest remaining ones into the selected
    if int(selected.sum()) != int(k):
        raise AssertionError(f"Hybrid selection count {selected.sum()} != {k}") # catch inconsistency before it reaches results
    return selected # done! return selected pool 

# this helper function is all about deriving metrics and results from our data and selections
def policy_metrics(frame, selected):

    # get numpy arrays from these columns for speed and operations (&)
    y = frame["TARGET"].to_numpy(dtype=int)
    w = frame["HBA1C_WEIGHT"].to_numpy(dtype=float)
    selected = np.asarray(selected, dtype=bool)

    selected_n = int(selected.sum()) # number of recalls
    total_positive = int(y.sum()) # total positive true cases 
    caught = int((selected & (y == 1)).sum()) # how many of the recalled were positive
    prevalence = float(y.mean()) # how many there are in total, ie this gives the base rate of what youd get picking randomly 

    # make stats
    precision = caught / selected_n if selected_n else np.nan
    recall = caught / total_positive if total_positive else np.nan

    # make weighted stats so its representative of the general population and what may happen if we used this extensively
    total_w = float(w.sum())
    selected_w = float(w[selected].sum())
    positive_w = float(w[y == 1].sum())
    caught_w = float(w[selected & (y == 1)].sum())
    weighted_prevalence = positive_w / total_w
    weighted_precision = caught_w / selected_w if selected_w else np.nan
    weighted_recall = caught_w / positive_w if positive_w else np.nan

    # under 50 caught metrics
    under50 = frame["AGE"].to_numpy(dtype=float) < 50
    under50_positive = (y == 1) & under50
    under50_caught = int((selected & under50_positive).sum())

    # return all of this rich data on metrics and perf stats
    return {
        "selected_n": selected_n,
        "sample_capacity_pct": 100 * selected_n / len(frame),
        "positive_selected_n": caught,
        "precision_pct": 100 * precision,
        "recall_pct": 100 * recall,
        "lift": precision / prevalence if prevalence else np.nan,
        "weighted_capacity_pct": 100 * selected_w / total_w,
        "weighted_precision_pct": 100 * weighted_precision,
        "weighted_recall_pct": 100 * weighted_recall,
        "weighted_lift": weighted_precision / weighted_prevalence if weighted_prevalence else np.nan,
        "under50_selected_n": int((selected & under50).sum()),
        "under50_positive_total_n": int(under50_positive.sum()),
        "under50_positive_caught_n": under50_caught,
        "under50_recall_pct": 100 * under50_caught / int(under50_positive.sum()) if under50_positive.sum() else np.nan,
    }

POLICIES = {
    "Global age + BMI": lambda frame, scores, k: global_policy(scores, k),
    "Age-stratified age + BMI": lambda frame, scores, k: age_stratified_policy(frame, scores, k),
    "Hybrid reserve 30% under-50": lambda frame, scores, k: hybrid_under50_policy(frame, scores, k, PRIMARY_RESERVE),
    "Hybrid reserve 45% under-50": lambda frame, scores, k: hybrid_under50_policy(frame, scores, k, SECONDARY_RESERVE),
}
print("Locked allocation policies defined.")

In [ ]:
### This cell is all about comparing the policies to see which was better

comparison_rows = []
policy_masks = {}

# nest loops for rows: 2 cycles * three capacities * four policies = 24 rows
for cycle_name in [VALID_CYCLE, TEST_CYCLE]:
    frame = cohorts[cycle_name]
    scores = score_frames[cycle_name]
    for capacity in [0.05, 0.10, 0.20]:
        k = int(np.ceil(len(frame) * capacity)) # convert the capacity from a percent/fraction to a number of slots, and round up
        for policy_name, policy_fn in POLICIES.items():
            mask = policy_fn(frame, scores, k)
            policy_masks[(cycle_name, capacity, policy_name)] = mask # stores masks in a dict for later use 
            comparison_rows.append({
                "cycle": cycle_name,
                "nominal_sample_capacity_pct": 100 * capacity,
                "policy": policy_name,
                **policy_metrics(frame, mask),
            })

policy_comparison = pd.DataFrame(comparison_rows)
primary_policy_results = policy_comparison[
    policy_comparison["nominal_sample_capacity_pct"].eq(10.0)
].copy()
display(primary_policy_results.round(3))
policy_comparison.to_csv(OUTPUT_DIR / "policy_comparison_validation_and_holdout.csv", index=False) # full table
primary_policy_results.to_csv(OUTPUT_DIR / "policy_comparison_10pct.csv", index=False) # 10% 

In [ ]:
### this cell checks if the model generalised and tests it with metrics
### the metrics have been explained above already in a md cell.
### unweighted = how well the model did on the sample
### weighted = how well the model did on the population the sample represents

model_metric_rows = []
for cycle_name in [VALID_CYCLE, TEST_CYCLE]:
    frame = cohorts[cycle_name]
    scores = score_frames[cycle_name]
    y = frame["TARGET"].to_numpy(dtype=int)
    w = frame["HBA1C_WEIGHT"].to_numpy(dtype=float)
    model_metric_rows.append({
        "cycle": cycle_name,
        "n": len(frame),
        "positive_n": int(y.sum()),
        "positive_pct_unweighted": 100 * y.mean(),
        "positive_pct_weighted": 100 * np.average(y, weights=w),
        "roc_auc_unweighted": roc_auc_score(y, scores),
        "roc_auc_weighted": roc_auc_score(y, scores, sample_weight=w),
        "average_precision_unweighted": average_precision_score(y, scores),
        "average_precision_weighted": average_precision_score(y, scores, sample_weight=w),
        "brier_unweighted": brier_score_loss(y, scores),
        "brier_weighted": brier_score_loss(y, scores, sample_weight=w),
    })

Results:

In [ ]:
for cycle_name in [VALID_CYCLE, TEST_CYCLE]:
    frame = cohorts[cycle_name]
    y = frame["TARGET"].to_numpy(dtype=int)
    for policy_name in POLICIES:
        mask = policy_masks[(cycle_name, 0.10, policy_name)]
        for dimension, column in [("Age", "AGE_GROUP"), ("Sex", "SEX"), ("Race/ethnicity", "ETHNICITY")]:
            for group in frame[column].astype("object").fillna("Unknown / missing").unique():
                group_mask = frame[column].astype("object").fillna("Unknown / missing").to_numpy() == group
                group_positive = group_mask & (y == 1)
                group_n = int(group_mask.sum())
                positive_n = int(group_positive.sum())
                selected_n = int((mask & group_mask).sum())
                caught_n = int((mask & group_positive).sum())
                subgroup_rows.append({
                    "cycle": cycle_name,
                    "policy": policy_name,
                    "dimension": dimension,
                    "group": str(group),
                    "n": group_n,
                    "positive_n": positive_n,
                    "selected_n": selected_n,
                    "selection_rate_pct": 100 * selected_n / group_n if group_n else np.nan,
                    "precision_pct": 100 * caught_n / selected_n if selected_n else np.nan,
                    "within_group_recall_pct": 100 * caught_n / positive_n if positive_n else np.nan,
                    "missed_positive_n": positive_n - caught_n,
                })